# Traductor Inglés → Español con Transformer (desde cero) — Dataset grande: OPUS-100

**Curso:** Tópicos Avanzados en Machine Learning — Proyecto de Neural Networks

Esta es la **misma arquitectura y el mismo pipeline** de `mt_en_es_transformer.ipynb` (Transformer
encoder-decoder implementado desde cero, mismos hiperparámetros), entrenada sobre un dataset **mucho más
grande**: [OPUS-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100) (par en-es), en vez de Tatoeba.
El objetivo es comparar el efecto de la **cantidad de datos** sobre la calidad de traducción, manteniendo
todo lo demás igual (arquitectura, hiperparámetros, número de épocas).

Genera archivos con sufijo `_opus100` para no pisar los del notebook de Tatoeba:
`transformer_en_es_opus100.pth`, `vocab_en_opus100.json`, `vocab_es_opus100.json`,
`model_config_opus100.json`, `history_opus100.json`, `resultados_test_opus100.csv`.

**Diseñado para Google Colab con GPU.** Por el tamaño del dataset, una corrida completa (15 épocas sobre
hasta 500,000 pares) toma aproximadamente 45-70 minutos en GPU T4 — más que el notebook de Tatoeba
(~16 min), porque hay bastante más data por época. Si te quedas corto de tiempo, baja `MAX_TRAIN_PAIRS`
o `N_EPOCHS` en la sección 6.

## 1. Setup e imports

In [ ]:
!pip install -q datasets

import os, re, json, math, time, random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Descarga de datos: OPUS-100 (en-es)

OPUS-100 es un corpus paralelo de ~55 idiomas curado por Helsinki-NLP, con oraciones de dominio mixto
(subtítulos, noticias, texto web) — mucho más grande y variado que Tatoeba. Lo cargamos desde el Hub de
Hugging Face (`datasets`), que ya trae splits de train/validation/test separados; no se usa ningún modelo
ni peso pre-entrenado, solo se descargan los pares de texto crudo.

In [ ]:
MAX_TRAIN_PAIRS = 500_000  # sube o baja esto según cuánto tiempo/GPU tengas disponible

raw = load_dataset('Helsinki-NLP/opus-100', 'en-es')
print(raw)

In [ ]:
def extract_pairs(split, limit=None):
    pairs = []
    for ex in split:
        tr = ex['translation']
        pairs.append((tr['en'], tr['es']))
        if limit is not None and len(pairs) >= limit:
            break
    return pairs

train_raw = extract_pairs(raw['train'], limit=MAX_TRAIN_PAIRS)
val_raw = extract_pairs(raw['validation'])
test_raw = extract_pairs(raw['test'])

print(f'Pares crudos -> train: {len(train_raw)}  val: {len(val_raw)}  test: {len(test_raw)}')

## 3. Limpieza, tokenización y vocabulario (desde cero)

Exactamente la misma normalización, tokenizador y clase `Vocab` que en el notebook de Tatoeba — así la
comparación entre datasets no se contamina con diferencias de preprocesamiento.

In [ ]:
MAX_LEN = 20
MIN_FREQ = 2

PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = '<pad>', '<sos>', '<eos>', '<unk>'

def normalize_text(s):
    s = s.strip().lower()
    s = re.sub(r"([.!?¿¡,])", r" \1 ", s)
    s = re.sub(r"[^a-zñáéíóúü¿¡.!?, ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def clean_pairs(raw_pairs):
    pairs = []
    for en, es in raw_pairs:
        en_n, es_n = normalize_text(en), normalize_text(es)
        if not en_n or not es_n:
            continue
        if len(en_n.split()) > MAX_LEN or len(es_n.split()) > MAX_LEN:
            continue
        pairs.append((en_n, es_n))
    return pairs

train_pairs = clean_pairs(train_raw)
val_pairs = clean_pairs(val_raw)
test_pairs = clean_pairs(test_raw)

print(f'Pares después de limpiar -> train: {len(train_pairs)}  val: {len(val_pairs)}  test: {len(test_pairs)}')
print('Ejemplo:', train_pairs[0])

In [ ]:
class Vocab:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}

    def build(self, sentences):
        counter = Counter()
        for s in sentences:
            counter.update(s.split())
        specials = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
        words = specials + [w for w, c in counter.items() if c >= self.min_freq]
        self.word2idx = {w: i for i, w in enumerate(words)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}

    def encode(self, sentence, add_sos_eos=True):
        ids = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in sentence.split()]
        if add_sos_eos:
            ids = [self.word2idx[SOS_TOKEN]] + ids + [self.word2idx[EOS_TOKEN]]
        return ids

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), UNK_TOKEN)
            if w == EOS_TOKEN:
                break
            if w in (SOS_TOKEN, PAD_TOKEN):
                continue
            words.append(w)
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)

src_vocab = Vocab(min_freq=MIN_FREQ)
src_vocab.build([p[0] for p in train_pairs])
tgt_vocab = Vocab(min_freq=MIN_FREQ)
tgt_vocab.build([p[1] for p in train_pairs])

print(f'Vocabulario inglés: {len(src_vocab)} palabras')
print(f'Vocabulario español: {len(tgt_vocab)} palabras')

# al menos 100 ejemplos de test, igual que en el notebook de Tatoeba
assert len(test_pairs) >= 100, 'el split de test quedó con menos de 100 ejemplos tras la limpieza'

## 4. Dataset y DataLoader

In [ ]:
PAD_IDX = src_vocab.word2idx[PAD_TOKEN]

class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_ids = torch.tensor(self.src_vocab.encode(src), dtype=torch.long)
        tgt_ids = torch.tensor(self.tgt_vocab.encode(tgt), dtype=torch.long)
        return src_ids, tgt_ids

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_pad = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=src_vocab.word2idx[PAD_TOKEN])
    tgt_pad = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=tgt_vocab.word2idx[PAD_TOKEN])
    return src_pad, tgt_pad

BATCH_SIZE = 128
train_loader = DataLoader(TranslationDataset(train_pairs, src_vocab, tgt_vocab), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(TranslationDataset(val_pairs, src_vocab, tgt_vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## 5. Arquitectura Transformer (desde cero)

Idéntica a la del notebook de Tatoeba: positional encoding senoidal, atención multi-cabeza manual,
feed-forward posicional, encoder y decoder con máscaras de padding y causal.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.w_q(q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.w_o(out)


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, src, mask):
        x = self.embed(src) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, mask)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, tgt, enc_out, src_mask, tgt_mask):
        x = self.embed(tgt) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.fc_out(x)


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, n_heads=8, d_ff=512,
                 n_layers=3, dropout=0.1, max_len=100, pad_idx=0):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.pad_idx = pad_idx

    def make_src_mask(self, src):
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        L = tgt.size(1)
        sub_mask = torch.tril(torch.ones((L, L), device=tgt.device)).bool()
        return pad_mask & sub_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask)
        return self.decoder(tgt, enc_out, src_mask, tgt_mask)

In [ ]:
D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT = 256, 8, 512, 3, 0.1
MAX_POS_LEN = 100  # mismo margen que en el notebook de Tatoeba (cubre train + generación en inferencia)

model = Transformer(len(src_vocab), len(tgt_vocab), d_model=D_MODEL, n_heads=N_HEADS,
                     d_ff=D_FF, n_layers=N_LAYERS, dropout=DROPOUT, max_len=MAX_POS_LEN,
                     pad_idx=PAD_IDX).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {n_params:,}')

## 6. Entrenamiento

Mismos hiperparámetros de optimización que en el notebook de Tatoeba (Adam, label smoothing, gradient
clipping, mismo `N_EPOCHS`), para que la única variable que cambia sea el dataset.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    with torch.set_grad_enabled(is_train):
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
N_EPOCHS = 15  # igual que en el notebook de Tatoeba; baja esto si te quedas corto de tiempo
CKPT_PATH = 'transformer_en_es_opus100.pth'

best_val = float('inf')
history = {'train_loss': [], 'val_loss': [], 'epoch_seconds': []}

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, model, optimizer)
    val_loss = run_epoch(val_loader, model)
    dt = time.time() - t0
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['epoch_seconds'].append(dt)
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), CKPT_PATH)
    print(f'Epoch {epoch:02d}/{N_EPOCHS} | train_loss {train_loss:.3f} | val_loss {val_loss:.3f} | {dt:.1f}s')

print('Mejor val_loss:', best_val)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.xlabel('Época'); plt.ylabel('Loss (CrossEntropy)'); plt.legend(); plt.title('Curva de entrenamiento — OPUS-100')
plt.show()

## 7. Inferencia (decodificación greedy)

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

def translate_sentence(sentence, model, src_vocab, tgt_vocab, max_len=30):
    model.eval()
    norm = normalize_text(sentence)
    src_ids = torch.tensor([src_vocab.encode(norm)], dtype=torch.long).to(device)
    src_mask = model.make_src_mask(src_ids)
    with torch.no_grad():
        enc_out = model.encoder(src_ids, src_mask)
    tgt_ids = [tgt_vocab.word2idx[SOS_TOKEN]]
    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        tgt_mask = model.make_tgt_mask(tgt_tensor)
        with torch.no_grad():
            out = model.decoder(tgt_tensor, enc_out, src_mask, tgt_mask)
        next_id = out[0, -1].argmax().item()
        tgt_ids.append(next_id)
        if next_id == tgt_vocab.word2idx[EOS_TOKEN]:
            break
    return tgt_vocab.decode(tgt_ids[1:])

for s in ['I love you.', 'What time is it?', 'The weather is nice today.']:
    print(f'{s!r:35s} -> {translate_sentence(s, model, src_vocab, tgt_vocab)}')

## 8. Evaluación en conjunto de test

Se usa el split de **test oficial de OPUS-100** (nunca visto en entrenamiento), con la misma métrica
BLEU (implementación propia) del notebook de Tatoeba.

In [ ]:
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def sentence_bleu(reference, hypothesis, max_n=4):
    ref_tokens, hyp_tokens = reference.split(), hypothesis.split()
    if len(hyp_tokens) == 0:
        return 0.0
    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = ngram_counts(ref_tokens, n)
        hyp_ngrams = ngram_counts(hyp_tokens, n)
        overlap = sum(min(c, ref_ngrams.get(g, 0)) for g, c in hyp_ngrams.items())
        total = max(sum(hyp_ngrams.values()), 1)
        precisions.append(overlap / total if total > 0 else 1e-9)
    if min(precisions) <= 0:
        geo_mean = 0.0
    else:
        geo_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)
    bp = 1.0 if len(hyp_tokens) > len(ref_tokens) else math.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1))
    return geo_mean * bp

In [ ]:
results = []
for src, tgt in test_pairs:
    hyp = translate_sentence(src, model, src_vocab, tgt_vocab)
    bleu = sentence_bleu(tgt, hyp)
    results.append({'ingles': src, 'espanol_referencia': tgt, 'espanol_predicho': hyp, 'bleu': bleu})

df = pd.DataFrame(results)
print(f"Tamaño del conjunto de test: {len(df)} ejemplos")
print(f"BLEU promedio en test: {df['bleu'].mean():.4f}")
df.to_csv('resultados_test_opus100.csv', index=False)

### Casos donde el modelo traduce bien (BLEU alto)

In [ ]:
pd.set_option('display.max_colwidth', None)
df.sort_values('bleu', ascending=False).head(15)[['ingles','espanol_referencia','espanol_predicho','bleu']]

### Casos donde el modelo falla (BLEU bajo)

In [ ]:
df.sort_values('bleu', ascending=True).head(15)[['ingles','espanol_referencia','espanol_predicho','bleu']]

## 9. Guardado de pesos, vocabularios e historial

Mismos artefactos que en el notebook de Tatoeba, con sufijo `_opus100` para no pisarlos — necesarios
para el notebook de comparación (`comparacion.ipynb`).

In [ ]:
with open('vocab_en_opus100.json', 'w', encoding='utf-8') as f:
    json.dump(src_vocab.word2idx, f, ensure_ascii=False)
with open('vocab_es_opus100.json', 'w', encoding='utf-8') as f:
    json.dump(tgt_vocab.word2idx, f, ensure_ascii=False)

config = {
    'd_model': D_MODEL, 'n_heads': N_HEADS, 'd_ff': D_FF, 'n_layers': N_LAYERS,
    'dropout': DROPOUT, 'max_len': MAX_POS_LEN,
}
with open('model_config_opus100.json', 'w') as f:
    json.dump(config, f)

history_out = dict(history)
history_out['dataset'] = 'OPUS-100 en-es'
history_out['n_train'] = len(train_pairs)
history_out['n_val'] = len(val_pairs)
history_out['n_test'] = len(test_pairs)
history_out['vocab_en'] = len(src_vocab)
history_out['vocab_es'] = len(tgt_vocab)
history_out['n_params'] = n_params
history_out['best_val_loss'] = best_val
history_out['test_bleu'] = float(df['bleu'].mean())
with open('history_opus100.json', 'w') as f:
    json.dump(history_out, f)

print('Guardado: transformer_en_es_opus100.pth, vocab_en_opus100.json, vocab_es_opus100.json,')
print('          model_config_opus100.json, history_opus100.json, resultados_test_opus100.csv')

## 10. Reporte parcial (esta corrida)

### 10.1 Dataset
- **Fuente:** [OPUS-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100) (par en-es), corpus
  paralelo de dominio mixto (subtítulos, noticias, web) curado por Helsinki-NLP.
- Mismo pipeline de limpieza y tokenización propia que en el notebook de Tatoeba.
- Tamaño usado en esta corrida: ver la celda de la sección 2/3 (`train_pairs`, `val_pairs`, `test_pairs`)
  y `history_opus100.json`.

### 10.2 Arquitectura
Exactamente la misma que en `mt_en_es_transformer.ipynb` (Transformer desde cero, `d_model=256`,
8 cabezas, 3+3 capas) — la única variable que cambia respecto a esa corrida es el dataset.

### 10.3 Resultados
*(Completar después de correr en Colab: BLEU promedio en test, ejemplos buenos y malos de las secciones
8 de este notebook.)* Para la comparación final con Tatoeba, usar `comparacion.ipynb`.